#### Name: Ripan Deep Kour
#### Student ID: N01454601

#### Import Libraries

In [1]:
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

import lab09_starter as S

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEEDS = [0, 1, 2]
BUDGET = S.BUDGET   # 20000
GAMMA = 0.99
PROBES = S.probe_states().to(DEVICE)

def summarize_episodes(episodes):
    curve = S.smooth_curve(episodes)
    return {
        "curve": curve,
        "final_return": S.final_return(episodes),
        "steps_to_150": S.steps_to_reach(curve, 150),
    }

In [3]:
def dqn_target(reward, next_q_max, gamma, terminated):
    # Only true termination stops bootstrapping; time-limit truncation still bootstraps.
    if terminated:
        return reward
    return reward + gamma * next_q_max

def huber_loss_manual(err, delta=1.0):
    ae = abs(err)
    if ae <= delta:
        return 0.5 * (ae ** 2)
    return delta * (ae - 0.5 * delta)

def ppo_clip_term(ratio, adv, eps=0.2):
    unclipped = ratio * adv
    clipped = np.clip(ratio, 1.0 - eps, 1.0 + eps) * adv
    return min(unclipped, clipped)

def returns_to_go(rewards, gamma):
    out = []
    running = 0.0
    for r in reversed(rewards):
        running = r + gamma * running
        out.append(running)
    out.reverse()
    return out

##### 1. Bootstrap cases (r=1, max_next=7, gamma=0.9)

In [4]:
print("--- 1. Bootstrap ---")
print("Terminal (True):   ", dqn_target(1.0, 7.0, 0.9, True))
print("Non-terminal:      ", dqn_target(1.0, 7.0, 0.9, False))
print("Truncated only:    ", dqn_target(1.0, 7.0, 0.9, False))

--- 1. Bootstrap ---
Terminal (True):    1.0
Non-terminal:       7.3
Truncated only:     7.3


##### 2. Huber loss at 0.5, 3, -3 (delta=1.0)

In [5]:
print("\n--- 2. Huber Loss ---")
huber_torch = nn.HuberLoss(delta=1.0, reduction="none")
for e in [0.5, 3.0, -3.0]:
    m_val = huber_loss_manual(e, delta=1.0)
    t_val = huber_torch(torch.tensor([e]), torch.tensor([0.0])).item()
    print(f"err={e:4.1f} | manual={m_val:.4f} | torch={t_val:.4f}")


--- 2. Huber Loss ---
err= 0.5 | manual=0.1250 | torch=0.1250
err= 3.0 | manual=2.5000 | torch=2.5000
err=-3.0 | manual=2.5000 | torch=2.5000


##### 3. Fixed-point check: 1 state, 1 action, r=1, next_state=state, never terminal, gamma=0.9 -> target=10.0

In [6]:
def run_fixed_point(use_target=True, updates=3000, sync_every=50, seed=0):
    S.set_seed(seed)
    s = torch.tensor([[0.1, -0.2, 0.05, 0.1]], dtype=torch.float32, device=DEVICE)
    q_net = S.QNet(obs_dim=4, n_actions=2).to(DEVICE)
    tgt_net = S.QNet(obs_dim=4, n_actions=2).to(DEVICE)
    tgt_net.load_state_dict(q_net.state_dict())
    opt = torch.optim.Adam(q_net.parameters(), lr=0.01)
    loss_fn = nn.HuberLoss(delta=1.0)

    for u in range(1, updates + 1):
        with torch.no_grad():
            ref_net = tgt_net if use_target else q_net
            next_v = ref_net(s)[:, 0]
            target = 1.0 + 0.9 * next_v
        pred = q_net(s)[:, 0]
        loss = loss_fn(pred, target)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if use_target and u % sync_every == 0:
            tgt_net.load_state_dict(q_net.state_dict())

    with torch.no_grad():
        return q_net(s)[:, 0].item()

print("\n--- 3. Fixed Point (true value = 10.0) ---")
print("With target network:    ", round(run_fixed_point(use_target=True), 4))
print("Without target network: ", round(run_fixed_point(use_target=False), 4))


--- 3. Fixed Point (true value = 10.0) ---
With target network:     9.9665
Without target network:  10.0


##### 4. PPO clipped objective (eps=0.2)

In [7]:
print("\n--- 4. PPO Clip (eps=0.2) ---")
for r in [1.3, 0.7]:
    for a in [1.0, -1.0]:
        print(f"ratio={r}, adv={a:+.0f} -> {ppo_clip_term(r, a, eps=0.2):.4f}")


--- 4. PPO Clip (eps=0.2) ---
ratio=1.3, adv=+1 -> 1.2000
ratio=1.3, adv=-1 -> -1.3000
ratio=0.7, adv=+1 -> 0.7000
ratio=0.7, adv=-1 -> -0.8000


##### 5. Return-to-go for [1, 1, 1], gamma=0.5

In [8]:
print("\n--- 5. Return-to-Go ---")
print("[1, 1, 1], gamma=0.5 ->", returns_to_go([1.0, 1.0, 1.0], 0.5))


--- 5. Return-to-Go ---
[1, 1, 1], gamma=0.5 -> [1.75, 1.5, 1.0]


#### Replay Buffer & DQN Agent

In [9]:
class ReplayBuffer:
    def __init__(self, capacity, mode="uniform"):
        self.capacity = capacity
        self.mode = mode
        self.data = []
        self.pos = 0

    def __len__(self):
        return len(self.data)

    def push(self, transition):
        if len(self.data) < self.capacity:
            self.data.append(transition)
        else:
            self.data[self.pos] = transition
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size):
        if self.mode == "recent_only":
            batch = self.data[-batch_size:]
        else:
            idxs = np.random.choice(len(self.data), size=batch_size, replace=False)
            batch = [self.data[i] for i in idxs]
        s, a, r, ns, done = zip(*batch)
        return (
            torch.tensor(np.array(s), dtype=torch.float32, device=DEVICE),
            torch.tensor(a, dtype=torch.long, device=DEVICE),
            torch.tensor(r, dtype=torch.float32, device=DEVICE),
            torch.tensor(np.array(ns), dtype=torch.float32, device=DEVICE),
            torch.tensor(done, dtype=torch.float32, device=DEVICE),
        )

def epsilon_by_step(step):
    if step >= 8000:
        return 0.05
    return 1.0 - (1.0 - 0.05) * (step / 8000.0)

def eval_q_stats(q_net):
    with torch.no_grad():
        q_vals = q_net(PROBES)
        mean_max_q = q_vals.max(dim=1).values.mean().item()
        max_abs_q = q_vals.abs().max().item()
    return mean_max_q, max_abs_q

def run_dqn(seed, use_target=True, buffer_capacity=20000, buffer_mode="uniform"):
    S.set_seed(seed)
    env = S.CartPole(seed=seed)
    q_net = S.QNet(obs_dim=4, n_actions=2).to(DEVICE)
    tgt_net = S.QNet(obs_dim=4, n_actions=2).to(DEVICE)
    tgt_net.load_state_dict(q_net.state_dict())

    optimizer = torch.optim.Adam(q_net.parameters(), lr=1e-3)
    loss_fn = nn.HuberLoss(delta=1.0)
    buf = ReplayBuffer(capacity=buffer_capacity, mode=buffer_mode)

    s = env.reset()
    ep_return = 0.0
    episodes = []
    q_probes_log = []   # (step, mean_max_q)
    peak_abs_q = 0.0

    start_after = 64 if buffer_mode == "recent_only" else 500
    batch_size = 64

    for step in range(1, BUDGET + 1):
        eps = epsilon_by_step(step)
        if np.random.rand() < eps:
            a = int(np.random.randint(2))
        else:
            with torch.no_grad():
                a = int(q_net(torch.tensor(s, dtype=torch.float32, device=DEVICE).unsqueeze(0)).argmax(dim=1).item())

        ns, r, term, trunc = env.step(a)
        # Store terminal ONLY for true termination
        buf.push((s, a, r, ns, float(term)))
        ep_return += r
        s = ns

        if term or trunc:
            episodes.append((step, ep_return))
            s = env.reset()
            ep_return = 0.0

        if step > start_after and len(buf) >= batch_size:
            bs, ba, br, bns, bdone = buf.sample(batch_size)
            with torch.no_grad():
                ref = tgt_net if use_target else q_net
                next_q_max = ref(bns).max(dim=1).values
                targets = br + GAMMA * (1.0 - bdone) * next_q_max

            q_pred = q_net(bs).gather(1, ba.unsqueeze(1)).squeeze(1)
            loss = loss_fn(q_pred, targets)

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(q_net.parameters(), 10.0)
            optimizer.step()

        if use_target and step % 250 == 0:
            tgt_net.load_state_dict(q_net.state_dict())

        if step % 1000 == 0:
            mean_mq, max_aq = eval_q_stats(q_net)
            q_probes_log.append((step, mean_mq))
            peak_abs_q = max(peak_abs_q, max_aq)

    stats = summarize_episodes(episodes)
    stats["q_probes"] = q_probes_log
    stats["peak_abs_q"] = peak_abs_q
    return stats

#### Run DQN & Ablations

In [10]:
configs_dqn = {
    "DQN (full)":      dict(use_target=True,  buffer_capacity=20000, buffer_mode="uniform"),
    "No target":       dict(use_target=False, buffer_capacity=20000, buffer_mode="uniform"),
    "No replay":       dict(use_target=True,  buffer_capacity=64,    buffer_mode="recent_only"),
    "Buffer 1,000":    dict(use_target=True,  buffer_capacity=1000,  buffer_mode="uniform"),
}

results = {}
for name, kw in configs_dqn.items():
    t0 = time.perf_counter()
    per_seed = {}
    for sd in SEEDS:
        per_seed[sd] = run_dqn(seed=sd, **kw)
    elapsed = time.perf_counter() - t0
    results[name] = {"seeds": per_seed, "time": elapsed}
    print(f"=== {name} ({elapsed:.2f}s) ===")
    for sd in SEEDS:
        m = per_seed[sd]
        print(f"  seed {sd}: final={m['final_return']:.1f}, step@150={m['steps_to_150']}, peak|Q|={m['peak_abs_q']:.2f}")

=== DQN (full) (352.05s) ===
  seed 0: final=25.5, step@150=None, peak|Q|=64.15
  seed 1: final=85.3, step@150=None, peak|Q|=60.70
  seed 2: final=114.1, step@150=8500, peak|Q|=59.76
=== No target (547.42s) ===
  seed 0: final=9.6, step@150=None, peak|Q|=88435224.00
  seed 1: final=9.6, step@150=None, peak|Q|=74567896.00
  seed 2: final=9.6, step@150=None, peak|Q|=73088840.00
=== No replay (495.74s) ===
  seed 0: final=145.7, step@150=8000, peak|Q|=61.06
  seed 1: final=139.6, step@150=None, peak|Q|=61.57
  seed 2: final=137.3, step@150=11500, peak|Q|=61.64
=== Buffer 1,000 (510.74s) ===
  seed 0: final=133.0, step@150=15000, peak|Q|=55.86
  seed 1: final=200.0, step@150=7000, peak|Q|=55.54
  seed 2: final=182.8, step@150=11500, peak|Q|=55.65


#### REINFORCE with Baseline & PPO

In [11]:
def run_reinforce(seed):
    t0 = time.perf_counter()
    S.set_seed(seed)
    env = S.CartPole(seed=seed)
    policy = S.mlp(4, 2).to(DEVICE)
    value_net = S.mlp(4, 1).to(DEVICE)

    opt_p = torch.optim.Adam(policy.parameters(), lr=3e-3)
    opt_v = torch.optim.Adam(value_net.parameters(), lr=1e-2)

    s = env.reset()
    ep_return = 0.0
    episodes = []
    batch_trajs = []  # list of (states, actions, rtgs)
    cur_s, cur_a, cur_r = [], [], []
    step = 0

    while step < BUDGET:
        while step < BUDGET:
            step += 1
            st = torch.tensor(s, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            logits = policy(st)
            dist = torch.distributions.Categorical(logits=logits)
            a = int(dist.sample().item())

            ns, r, term, trunc = env.step(a)
            cur_s.append(s)
            cur_a.append(a)
            cur_r.append(r)
            ep_return += r
            s = ns

            if term or trunc or step == BUDGET:
                rtg = returns_to_go(cur_r, GAMMA)
                batch_trajs.append((cur_s, cur_a, rtg))
                if term or trunc:
                    episodes.append((step, ep_return))
                    s = env.reset()
                    ep_return = 0.0
                cur_s, cur_a, cur_r = [], [], []
                if len(batch_trajs) == 8 or step == BUDGET:
                    break

        if not batch_trajs:
            break

        all_s = torch.tensor(np.array([x for tr in batch_trajs for x in tr[0]]), dtype=torch.float32, device=DEVICE)
        all_a = torch.tensor([x for tr in batch_trajs for x in tr[1]], dtype=torch.long, device=DEVICE)
        all_rtg = torch.tensor([x for tr in batch_trajs for x in tr[2]], dtype=torch.float32, device=DEVICE)

        with torch.no_grad():
            v_pred = value_net(all_s).squeeze(-1)
        adv = all_rtg - v_pred
        adv = (adv - adv.mean()) / (adv.std() + 1e-8)

        logits = policy(all_s)
        log_probs = torch.distributions.Categorical(logits=logits).log_prob(all_a)
        policy_loss = -(log_probs * adv).mean()

        opt_p.zero_grad()
        policy_loss.backward()
        opt_p.step()

        v_out = value_net(all_s).squeeze(-1)
        value_loss = F.mse_loss(v_out, all_rtg)

        opt_v.zero_grad()
        value_loss.backward()
        opt_v.step()

        batch_trajs = []

    out = summarize_episodes(episodes)
    out["time"] = time.perf_counter() - t0
    return out

def compute_gae(rewards, values, dones, next_value, gamma=0.99, lam=0.95):
    advs = np.zeros_like(rewards, dtype=np.float32)
    gae = 0.0
    for t in reversed(range(len(rewards))):
        non_term = 1.0 - dones[t]
        nv = next_value if t == len(rewards) - 1 else values[t + 1]
        delta = rewards[t] + gamma * nv * non_term - values[t]
        gae = delta + gamma * lam * non_term * gae
        advs[t] = gae
    returns = advs + values
    return advs, returns

def run_ppo(seed, rollout_len=1024, epochs=6, minibatch_size=128, clip_eps=0.2, vf_coef=0.5):
    t0 = time.perf_counter()
    S.set_seed(seed)
    env = S.CartPole(seed=seed)
    policy = S.mlp(4, 2).to(DEVICE)
    value_net = S.mlp(4, 1).to(DEVICE)
    optimizer = torch.optim.Adam(list(policy.parameters()) + list(value_net.parameters()), lr=3e-4)

    s = env.reset()
    ep_return = 0.0
    episodes = []
    step = 0

    while step < BUDGET:
        obs_buf, act_buf, logp_buf, rew_buf, done_buf, val_buf = [], [], [], [], [], []
        n_collect = min(rollout_len, BUDGET - step)

        for _ in range(n_collect):
            step += 1
            st = torch.tensor(s, dtype=torch.float32, device=DEVICE).unsqueeze(0)
            with torch.no_grad():
                logits = policy(st)
                dist = torch.distributions.Categorical(logits=logits)
                a = int(dist.sample().item())
                lp = float(dist.log_prob(torch.tensor(a, device=DEVICE)).item())
                v = float(value_net(st).item())

            ns, r, term, trunc = env.step(a)
            obs_buf.append(s)
            act_buf.append(a)
            logp_buf.append(lp)
            rew_buf.append(r)
            done_buf.append(float(term))  # True termination only
            val_buf.append(v)

            ep_return += r
            s = ns
            if term or trunc:
                episodes.append((step, ep_return))
                s = env.reset()
                ep_return = 0.0

        with torch.no_grad():
            next_v = float(value_net(torch.tensor(s, dtype=torch.float32, device=DEVICE).unsqueeze(0)).item())

        obs_arr = np.array(obs_buf, dtype=np.float32)
        act_arr = np.array(act_buf, dtype=np.int64)
        old_logp_arr = np.array(logp_buf, dtype=np.float32)
        rew_arr = np.array(rew_buf, dtype=np.float32)
        done_arr = np.array(done_buf, dtype=np.float32)
        val_arr = np.array(val_buf, dtype=np.float32)

        adv_arr, ret_arr = compute_gae(rew_arr, val_arr, done_arr, next_v, gamma=GAMMA, lam=0.95)
        adv_arr = (adv_arr - adv_arr.mean()) / (adv_arr.std() + 1e-8)

        b_obs = torch.tensor(obs_arr, device=DEVICE)
        b_act = torch.tensor(act_arr, device=DEVICE)
        b_old_lp = torch.tensor(old_logp_arr, device=DEVICE)
        b_adv = torch.tensor(adv_arr, device=DEVICE)
        b_ret = torch.tensor(ret_arr, device=DEVICE)

        n_samples = len(obs_arr)
        for _ in range(epochs):
            perm = np.random.permutation(n_samples)
            for start in range(0, n_samples, minibatch_size):
                mb = perm[start:start + minibatch_size]
                logits = policy(b_obs[mb])
                dist = torch.distributions.Categorical(logits=logits)
                new_lp = dist.log_prob(b_act[mb])
                ratio = torch.exp(new_lp - b_old_lp[mb])

                surr1 = ratio * b_adv[mb]
                surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * b_adv[mb]
                pi_loss = -torch.min(surr1, surr2).mean()

                v_pred = value_net(b_obs[mb]).squeeze(-1)
                v_loss = F.mse_loss(v_pred, b_ret[mb])

                loss = pi_loss + vf_coef * v_loss
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(list(policy.parameters()) + list(value_net.parameters()), 0.5)
                optimizer.step()

    out = summarize_episodes(episodes)
    out["time"] = time.perf_counter() - t0
    return out

for pg_name, fn in [("REINFORCE + baseline", run_reinforce), ("PPO", run_ppo)]:
    per_seed = {}
    for sd in SEEDS:
        per_seed[sd] = fn(sd)
    results[pg_name] = {"seeds": per_seed, "time": sum(per_seed[sd]["time"] for sd in SEEDS)}
    print(f"=== {pg_name} (total 3-seed time: {results[pg_name]['time']:.2f}s) ===")
    for sd in SEEDS:
        m = per_seed[sd]
        print(f"  seed {sd}: final={m['final_return']:.1f}, step@150={m['steps_to_150']}, time={m['time']:.2f}s")

=== REINFORCE + baseline (total 3-seed time: 40.40s) ===
  seed 0: final=148.6, step@150=11000, time=19.32s
  seed 1: final=171.5, step@150=11000, time=11.07s
  seed 2: final=145.1, step@150=15000, time=10.00s
=== PPO (total 3-seed time: 64.19s) ===
  seed 0: final=170.3, step@150=15000, time=21.73s
  seed 1: final=194.5, step@150=11000, time=21.52s
  seed 2: final=176.3, step@150=11000, time=20.94s


#### All-Six-Agent Curves, Q-Value Plot, and Table

##### 1. Six-panel learning curves (every seed drawn, mean in bold)

In [12]:
fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True, sharey=True)
colors = ["#1C3D6E", "#B22222", "#D97706", "#2E7D32", "#6A1B9A", "#00838F"]

for ax, (name, data), col in zip(axes.flat, results.items(), colors):
    curves = {sd: data["seeds"][sd]["curve"] for sd in SEEDS}
    S.plot_curves(ax, curves, name, color=col)

fig.tight_layout()
fig.savefig("all_agents_curves.png", dpi=150)

##### 2. Q-value probe curves vs theoretical max bound of 100

In [13]:
fig2, ax2 = plt.subplots(figsize=(7, 4))
for name in ["DQN (full)", "No target", "No replay", "Buffer 1,000"]:
    steps_x = [pt[0] for pt in results[name]["seeds"][0]["q_probes"]]
    means_per_seed = np.array([[pt[1] for pt in results[name]["seeds"][sd]["q_probes"]] for sd in SEEDS])
    ax2.plot(steps_x, means_per_seed.mean(axis=0), label=name, lw=2)

ax2.axhline(100.0, color="black", linestyle="--", label="Theoretical bound (100)")
ax2.set_title("Mean Max Q on Probe States (3-seed mean)")
ax2.set_xlabel("environment steps")
ax2.set_ylabel("mean max Q")
ax2.legend()
ax2.grid(alpha=0.25)
fig2.tight_layout()
fig2.savefig("q_values_vs_bound.png", dpi=150)

##### 3. Print per-seed summary table

In [14]:
print("\nAgent | Seed | Final Return | Steps to 150 | Wall Time (s)")
for name, data in results.items():
    for sd in SEEDS:
        m = data["seeds"][sd]
        t_val = f"{m['time']:.2f}" if "time" in m else "-"
        print(f"{name:20s} | {sd} | {m['final_return']:12.1f} | {str(m['steps_to_150']):12s} | {t_val}")


Agent | Seed | Final Return | Steps to 150 | Wall Time (s)
DQN (full)           | 0 |         25.5 | None         | -
DQN (full)           | 1 |         85.3 | None         | -
DQN (full)           | 2 |        114.1 | 8500         | -
No target            | 0 |          9.6 | None         | -
No target            | 1 |          9.6 | None         | -
No target            | 2 |          9.6 | None         | -
No replay            | 0 |        145.7 | 8000         | -
No replay            | 1 |        139.6 | None         | -
No replay            | 2 |        137.3 | 11500        | -
Buffer 1,000         | 0 |        133.0 | 15000        | -
Buffer 1,000         | 1 |        200.0 | 7000         | -
Buffer 1,000         | 2 |        182.8 | 11500        | -
REINFORCE + baseline | 0 |        148.6 | 11000        | 19.32
REINFORCE + baseline | 1 |        171.5 | 11000        | 11.07
REINFORCE + baseline | 2 |        145.1 | 15000        | 10.00
PPO                  | 0 |        170.3 | 1